# 01 — Attention Mechanics

Attention is the *communication* step of a transformer: each position
builds a query, scores it against every key, and takes a probability-
weighted average of values. This notebook visualises those weights and
verifies the causal mask directly.

In [ ]:
import sys
sys.path.insert(0, "..")  # run from the notebooks/ directory

import matplotlib
import torch

import kamui
from kamui.model.config import ModelConfig

torch.manual_seed(0)
print("KAMUI", kamui.__version__)

In [ ]:
from kamui.model.attention import MultiHeadAttention, scaled_dot_product_attention

# The core function: softmax(QK^T / sqrt(d_k)) V, weights always returned.
q = k = v = torch.randn(1, 1, 5, 8)
out, weights = scaled_dot_product_attention(q, k, v)
print("weights rows sum to 1:", torch.allclose(weights.sum(-1), torch.ones(1, 1, 5)))

In [ ]:
# Causal masking: -inf before softmax means EXACTLY zero attention to the future.
mask = torch.triu(torch.ones(5, 5, dtype=torch.bool), diagonal=1)
_, weights = scaled_dot_product_attention(q, k, v, mask=mask)
print(weights[0, 0].round(decimals=2))

In [ ]:
# Train a tiny model on a synthetic corpus (~30s on CPU).
# The corpus is a seeded word-salad: repetitive enough to learn, varied enough
# that BPE cannot collapse it into a handful of giant tokens.
import random

from kamui.tokenizer.bpe import BPETokenizer
from kamui.training import DataLoader, TextDataset, Trainer, TrainingConfig

rng = random.Random(0)
WORDS = ["the", "cat", "dog", "sat", "ran", "on", "to", "mat", "log", "sun"]
CORPUS = " ".join(rng.choice(WORDS) for _ in range(4000))

config = ModelConfig(n_layers=2, d_model=64, n_heads=4, d_ff=128,
                     vocab_size=300, context_length=32, dropout=0.0)
tokenizer = BPETokenizer.train(CORPUS, vocab_size=config.vocab_size)
tokens = tokenizer.encode(CORPUS)

model = kamui.KAMUITransformer(config)
trainer = Trainer(
    model,
    DataLoader(TextDataset(tokens, config.context_length), batch_size=8, seed=0),
    config=TrainingConfig(max_lr=3e-3, warmup_steps=10, max_steps=1000),
)
records = trainer.train(150)
model.eval()
print(f"loss: {records[0]['train_loss']:.3f} -> {records[-1]['train_loss']:.3f}")

In [ ]:
# Visualise every head of the trained model on a real prompt.
from kamui.mechinterp import AttentionVisualizer, head_summary_stats

ids = torch.tensor(tokenizer.encode("the cat sat on the mat"))
result = AttentionVisualizer(model, tokenizer).run(ids)
result.plot_all()

In [ ]:
# Per-head statistics reveal head *types*: sharp heads, previous-token heads...
stats = head_summary_stats(result)
for i in range(len(stats["layer"])):
    print(f"L{stats['layer'][i]}H{stats['head'][i]}  "
          f"entropy={stats['entropy'][i]:.2f}  "
          f"self={stats['self_frac'][i]:.2f}  prev={stats['prev_frac'][i]:.2f}")